# Notebook Kaggle : Pipeline NLP CREMMA Medieval

Ce notebook clone le code de la branche `nlp-pipeline`, recupere les donnees HTR depuis S3,
puis execute la pipeline NLP reelle via `nlp_pipeline/nlp_cli.py` :
validate -> eda -> review-queue -> normalize-contract -> correct -> relative-eval -> lexical-check -> split.

## 1. Cloner le repo et installer les dependances

In [ ]:
import os

repo_dir = "htr-cremma-medieval-2026"
if not os.path.isdir(repo_dir):
    !git clone https://github.com/Loulou441/htr-cremma-medieval-2026.git

os.chdir(repo_dir)
!git checkout nlp-pipeline
!git pull
!pip install -q jsonschema transformers sentencepiece boto3

## 2. Recuperer les donnees HTR depuis S3

Les sorties HTR (`data/nlp_output/`) et le lexique sont stockes sur `s3://htr-cremma-medieval/nlp/`.
Renseignez vos credentials AWS dans les **Kaggle Secrets** (Add-ons > Secrets) sous les noms
`AWS_ACCESS_KEY_ID` et `AWS_SECRET_ACCESS_KEY`.

In [ ]:
!pip install -q awscli

import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "eu-west-3"

!aws s3 sync s3://htr-cremma-medieval/nlp/output/ data/nlp_output/
!aws s3 sync s3://htr-cremma-medieval/nlp/lexique/ data/lexique/
!aws s3 cp s3://htr-cremma-medieval/nlp/dictionary/dictionnaire_ancien_francais.json data/dictionary/dictionnaire_ancien_francais.json

## 3. Verifier que le CLI est accessible

In [ ]:
from pathlib import Path
import os

# Remonte a la racine du repo si le kernel demarre dans notebooks/ (cas VSCode local)
if Path("nlp_pipeline").is_dir():
    project_root = Path(".").resolve()
elif Path("../nlp_pipeline").is_dir():
    os.chdir("..")
    project_root = Path(".").resolve()
else:
    raise FileNotFoundError("nlp_pipeline/ introuvable depuis le working directory courant")

cli_path = project_root / "nlp_pipeline" / "nlp_cli.py"
print("Project root:", project_root)
print("CLI exists:", cli_path.exists())

sample_docs = sorted((project_root / "data" / "nlp_output").rglob("*.json"))
print("Sample HTR contracts found:", len(sample_docs))
sample_doc = sample_docs[0] if sample_docs else None
print("Using sample:", sample_doc)

## 4. Lancer la pipeline NLP etape par etape

Chaque commande est appelee en sous-processus pour eviter tout souci de `sys.path`.

In [ ]:
import subprocess


def run_cli(*args):
    cmd = ["python", str(cli_path), *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("$ ", " ".join(cmd))
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

In [ ]:
# 4.1 Validation du contrat HTR
run_cli("validate", "--input", str(sample_doc))

In [ ]:
# 4.2 EDA : statistiques de confiance et longueur de ligne
run_cli("eda", "--input", str(sample_doc), "--output", "data/eda_report.json")

In [ ]:
# 4.3 File de relecture : tri par confiance
run_cli(
    "review-queue",
    "--input", str(sample_doc),
    "--csv-output", "data/review/review_queue.csv",
    "--json-output", "data/review/review_buckets.json",
)

In [ ]:
# 4.4 Normalisation du francais medieval (u/v, i/j, abreviations)
run_cli(
    "normalize-contract",
    "--input", str(sample_doc),
    "--output", "data/contracts/contract_normalized.json",
)

In [ ]:
# 4.5 Correction guidee par confiance (heuristique par defaut, sans MLM)
run_cli(
    "correct",
    "--input", str(sample_doc),
    "--output", "data/contracts/contract_corrected.json",
    "--log-output", "data/review/correction_log.jsonl",
)

In [ ]:
# 4.5bis Variante avec modele de langue masque CamemBERT (plus lent, GPU recommande)
# run_cli(
#     "correct",
#     "--input", str(sample_doc),
#     "--output", "data/contracts/contract_corrected_mlm.json",
#     "--log-output", "data/review/correction_log_mlm.jsonl",
#     "--mlm-model", "almanach/camembert-base",
#     "--mlm-device", "auto",
# )

## 5. Evaluation relative (sans verite terrain)

`data/nlp_output/` ne contient que des predictions HTR brutes, pas de transcription
validee manuellement -- comparer "avant/apres" a une fausse reference (le texte brut
lui-meme) n'a pas de sens (CER avant = 0 par construction).

A la place, `relative-eval` calcule le **CER pairwise moyen** entre plusieurs variantes
d'une meme ligne (`raw`, `text_normalized`, `corrected`). Plus les variantes sont
proches les unes des autres, plus le pipeline est stable ; un ecart important signale
que la normalisation/correction modifie beaucoup le texte (a verifier manuellement
si c'est une amelioration ou une derive).

In [ ]:
import csv
import json

raw_contract = json.loads(Path(sample_doc).read_text(encoding="utf-8"))
normalized_contract = json.loads(Path("data/contracts/contract_normalized.json").read_text(encoding="utf-8"))
corrected_contract = json.loads(Path("data/contracts/contract_corrected.json").read_text(encoding="utf-8"))

raw_lines = [line["text"] for page in raw_contract.get("pages", []) for line in page.get("lines", [])]
normalized_lines = [
    line.get("normalized_text", line["text"])
    for page in normalized_contract.get("pages", [])
    for line in page.get("lines", [])
]
corrected_lines = [line["text"] for page in corrected_contract.get("pages", []) for line in page.get("lines", [])]

rows = [
    {"raw": raw, "text_normalized": norm, "corrected": corr}
    for raw, norm, corr in zip(raw_lines, normalized_lines, corrected_lines)
]
relative_csv = Path("data/review/relative_eval_sample.csv")
with relative_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["raw", "text_normalized", "corrected"])
    writer.writeheader()
    writer.writerows(rows)

run_cli("relative-eval", "--csv-input", str(relative_csv))

## 5bis. Detection lexicale via le dictionnaire d'ancien francais

Contrairement a `detect-normalization` (qui repere seulement des marqueurs
d'abreviation typographiques comme `~`, `⁊`, `ꝑ`), `lexical-check` verifie si
chaque token existe dans `dictionnaire_ancien_francais.json` (Wiktionary +
lexique CLTK). Les tokens absents sont de vraies erreurs lexicales potentielles
(mots mal transcrits, abreviations non resolues, mots inconnus du corpus).

In [ ]:
run_cli(
    "lexical-check",
    "--dictionary", "data/dictionary/dictionnaire_ancien_francais.json",
    "--output-dir", "data/nlp_output",
    "--top-n", "30",
    "--output", "data/lexical_report.json",
)

## 6. Push des resultats vers S3 (optionnel)

In [ ]:
sync_script = project_root / "nlp_pipeline" / "sync_to_s3.py"
subprocess.run(["python", str(sync_script)], capture_output=True, text=True)